## PIPELINE OF A FORECASTING MODEL WITH AN INCIDENCE/ONSET VARIABLE

### 1. LOAD DATA AND SET INDEX

In [ ]:
import pandas as pd
import numpy as np

# Load your dataset
df = pd.read_csv('your_dataset.csv')

# Ensure the time column is in datetime format
df['month'] = pd.to_datetime(df['month'])

# Set multi-level index (Entity/Region and Time)
# Essential for panel data handling and the PanelSplit logic
df = df.set_index(['country', 'month']).sort_index()

### 2. CREATE TARGET VARIABLE

It's going to be a normal column computed from values of the future. Ex: 

In [ ]:
# Define the forecasting horizon (e.g., predicting 2 months ahead)
horizon = 2 

# Create the target variable: 1 if conflict occurs in the future, 0 otherwise
# Using groupby to ensure shifts stay within the same geographical entity
df['target'] = (df.groupby(level='country')['conflict_variable']
                .shift(-horizon) > 0).astype(int)

# Drop the last rows of each group where the target is NaN (no future data available)
df = df.dropna(subset=['target'])

### 3. FEATURE ENGINEERING

If we are doing forecasting in t+h, then our features can only contain information since t (included).

In [ ]:
# Generate Lags (Historical values)
for lag in [1, 2, 3]:
    df[f'feature_lag{lag}'] = df.groupby(level='country')['some_variable'].shift(lag)

# Generate Rolling Windows (Moving averages/sums)
df['rolling_mean_3m'] = (df.groupby(level='country')['some_variable']
                         .transform(lambda x: x.rolling(window=3).mean()))

# Handle missing values generated by lagging
df = df.fillna(0)

### 4. CREATE X AND y

In [ ]:
# Delete all missing values

# Define the subset of columns to be used as predictors
# You can group your PCA, Signals, and Historical features here
features = ['feature_lag1', 'feature_lag2', 'rolling_mean_3m']

df_clean = df[features].dropna()

X = df[features]
y = df['target']

### 5. DEFINE THE PanelSplit

In [ ]:
import PanelSplit

# The periods variable identifies the time dimension for the walk-forward validation
periods = X.index.get_level_values('month')

# Initialize the custom PanelSplitter
panel_split = PanelSplit(
    periods=periods,
    n_splits=24,   # Number of validation folds
    test_size=1,   # Number of months to test in each fold
    gap=horizon - 1  # Prevents data leakage by ensuring the gap >= lead_time
)

### 6. DEFINE THE MODEL

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the classifier
# Using 'balanced' class_weight is critical for rare event detection (like conflict onset)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

### 7. APPLY CROSS_VAL_FIT_PREDICT

In [ ]:
all_y_true = []
all_y_pred = []
all_y_prob = []

# Walk-forward cross-validation loop
for train_idx, test_idx in panel_split.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Train the model
    rf_model.fit(X_train, y_train)
    
    # Generate predictions (probabilities and hard classes)
    probs = rf_model.predict_proba(X_test)[:, 1]
    preds = rf_model.predict(X_test)
    
    all_y_true.extend(y_test.values)
    all_y_pred.extend(preds)
    all_y_prob.extend(probs)

### 8. EVALUATION WITH METRICS

Important to do both Roc curve and Precision-Recall curve.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             average_precision_score, RocCurveDisplay, 
                             PrecisionRecallDisplay)

# 1. Setup the figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- PLOT 1: ROC CURVE ---
fpr, tpr, _ = roc_curve(all_y_true, all_y_prob)
roc_auc = auc(fpr, tpr)

ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Diagonal random line
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# --- PLOT 2: PRECISION-RECALL CURVE ---
precision, recall, _ = precision_recall_curve(all_y_true, all_y_prob)
avg_precision = average_precision_score(all_y_true, all_y_prob)

ax2.plot(recall, precision, color='blue', lw=2, label=f'Avg Precision = {avg_precision:.2f}')
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision (Positive Predictive Value)')
ax2.set_title('Precision-Recall (PR) Curve')
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 2. Print Summary Statistics
print(f"Final AUC-ROC: {roc_auc:.4f}")
print(f"Final Average Precision: {avg_precision:.4f}")